# TODO:

    1. Research which loss function to use
    2. Research which optimization function to use
    3. Research which model will be best to use

In [1]:
from pathlib import Path
import polars as pl

In [2]:
#----------------------------------------------------------------------
# Configuration
# ----------------------------------------------------------------------
data_path = "../data/"
is_training_data = True

if is_training_data:
    saving_path = data_path + "Final Training Data/"
    working_data_path = data_path + "Training Data/"
else:
    saving_path = data_path + "Final Data For Labeling/"
    working_data_path = data_path + "Data For Labeling/"

Path(saving_path).mkdir(parents=True, exist_ok=True)

In [3]:
# ----------------------------------------------------------------------
# Helper: read a single CSV file or all CSVs for a recording
# ----------------------------------------------------------------------
def read_csv(path: Path, schema_overrides=None):
    return pl.scan_csv(path, schema_overrides=schema_overrides)

In [4]:
# ----------------------------------------------------------------------
# Process labels per recording (the same logic, but we'll apply per recording)
# ----------------------------------------------------------------------
def process_labels_for_recording(lf: pl.LazyFrame) -> pl.LazyFrame:
    """Process labels for a single recording (already filtered by recording id)."""
    return (
        lf
        .sort("start timestamp [ms]")
        .with_columns([
            pl.col("label").shift(1).alias("_prev_label"),
            pl.col("end timestamp [ms]").shift(1).alias("_prev_end"),
        ])
        .with_columns(
            (
                (pl.col("label") != pl.col("_prev_label"))
                | (pl.col("start timestamp [ms]") != pl.col("_prev_end"))
            )
            .cast(pl.Int64)
            .cum_sum()
            .alias("group")
        )
        .group_by("group")
        .agg([
            pl.col("start timestamp [ms]").first(),
            pl.col("end timestamp [ms]").last(),
            pl.col("label").first(),
        ])
        .drop("group")
    )


In [5]:
# ----------------------------------------------------------------------
# Label gaze for a recording
# ----------------------------------------------------------------------
def label_gaze_for_recording(gaze: pl.LazyFrame, labels: pl.LazyFrame) -> pl.LazyFrame:
    gaze = gaze.with_columns(
        ((pl.col("timestamp [ns]") - pl.col("timestamp [ns]").min()) / 1_000_000)
        .cast(pl.Int64)
        .alias("time [ms]")
    )
    gaze = gaze.sort("time [ms]")
    labels = labels.sort("start timestamp [ms]")

    joined = gaze.join_asof(
        labels,
        left_on="time [ms]",
        right_on="start timestamp [ms]",
        strategy="backward",
    )
    return joined.filter(pl.col("time [ms]") <= pl.col("end timestamp [ms]"))

In [6]:
def process_recording(rec_dir: Path, is_training: bool, output_dir: Path):
    rec_id = rec_dir.name

    # 1. Read all relevant CSVs for this recording
    gaze_lf = read_csv(rec_dir / "gaze.csv", schema_overrides={"fixation id": pl.Utf8, "blink id": pl.Utf8, "saccades id": pl.Utf8})
    saccades_lf = read_csv(rec_dir / "saccades.csv", schema_overrides={"saccades id": pl.Utf8})
    fixations_lf = read_csv(rec_dir / "fixations.csv", schema_overrides={"fixation id": pl.Utf8})
    blinks_lf = read_csv(rec_dir / "blinks.csv", schema_overrides={"blink id": pl.Utf8})
    imu_lf = read_csv(rec_dir / "imu.csv")
    eye3d_lf = read_csv(rec_dir / "3d_eye_states.csv")
    if is_training:
        labels_lf = read_csv(rec_dir / "labels.csv")
    else:
        labels_lf = None

    # 2. Filter and drop early
    gaze_lf = gaze_lf.filter(pl.col("worn") != 0)
    gaze_lf = gaze_lf.drop([
        "worn",
        "gaze mono left x [px]",
        "gaze mono left y [px]",
        "gaze mono right x [px]",
        "gaze mono right y [px]",
    ])

    # 3. Process labels if training
    if is_training and labels_lf is not None:
        labels_lf = process_labels_for_recording(labels_lf)
        gaze_lf = label_gaze_for_recording(gaze_lf, labels_lf)
        gaze_lf =gaze_lf.filter(pl.col("label") != "Undefined")

    # 4. Join with saccades
    saccades_lf = saccades_lf.drop("section id")
    gaze_lf = gaze_lf.sort("timestamp [ns]")
    saccades_lf = saccades_lf.sort("start timestamp [ns]")
    gaze_lf = gaze_lf.join_asof(
        saccades_lf,
        left_on="timestamp [ns]",
        right_on="start timestamp [ns]",
        strategy="backward",
    ).filter(pl.col("timestamp [ns]") <= pl.col("end timestamp [ns]"))

    # 5. Join fixations and blinks
    fix_cols = ["fixation id", "duration [ms]", "fixation x [px]", "fixation y [px]"]
    sacc_cols = ["saccade id", "duration [ms]", "amplitude [deg]", "mean velocity [px/s]", "peak velocity [px/s]"]
    blink_cols = ["blink id", "duration [ms]"]

    gaze_lf = gaze_lf.join(fixations_lf.select(fix_cols), on="fixation id", how="left", suffix=" fix")
    gaze_lf = gaze_lf.join(saccades_lf.select(sacc_cols), on="saccade id", how="left", suffix=" sacc")
    gaze_lf = gaze_lf.join(blinks_lf.select(blink_cols), on="blink id", how="left", suffix=" blink")

    # 6. Merge IMU and 3D
    imu_lf = imu_lf.sort("timestamp [ns]")
    eye3d_lf = eye3d_lf.sort("timestamp [ns]")
    gaze_lf = gaze_lf.sort("timestamp [ns]")
    gaze_lf = gaze_lf.join_asof(imu_lf, on="timestamp [ns]", strategy="nearest", tolerance=50_000_000, suffix=" imu")
    gaze_lf = gaze_lf.join_asof(eye3d_lf, on="timestamp [ns]", strategy="nearest", tolerance=50_000_000, suffix=" eye3d")

    # 7. Drop extra columns
    gaze_lf = gaze_lf.drop([
        "section id imu", "recording id imu",
        "section id eye3d", "recording id eye3d",
    ], strict=False)

    # 8. Cast to Float32
    float_cols = [
        "gaze x [px]", "gaze y [px]",
        "fixation x [px]", "fixation y [px]",
        "gyro x [deg/s]", "gyro y [deg/s]", "gyro z [deg/s]",
        "acceleration x [g]", "acceleration y [g]", "acceleration z [g]",
        "duration [ms]", "duration [ms] fix", "duration [ms] sacc", "duration [ms] blink",
    ]
    gaze_lf = gaze_lf.with_columns([pl.col(c).cast(pl.Float32) for c in float_cols])

    # 9. Fill nulls
    gaze_lf = gaze_lf.with_columns([
        pl.col("fixation id").fill_null("unknown"),
        pl.col("blink id").fill_null("unknown"),
    ])
    fill_cols = {
        "duration [ms] blink": -1.0,
        "fixation y [px]": -1.0,
        "fixation x [px]": -1.0,
        "duration [ms] fix": -1.0,
    }
    gaze_lf = gaze_lf.with_columns([pl.col(c).fill_null(v) for c, v in fill_cols.items()])

    nan_cols = [
        col
        for col, dtype in gaze_lf.collect_schema().items()
        if dtype.is_float()
        and gaze_lf.select(pl.col(col).is_nan().any()).collect().item()
    ]
    
    print(f"Cols with NaNs: {nan_cols}")
    # 10. Write to per‑recording Parquet
    rec_output_path = output_dir / f"{rec_id}.parquet"
    gaze_lf.sink_parquet(rec_output_path)

    print(f"Processed recording: {rec_id} -> {rec_output_path.name}")

In [8]:
output_dir = Path(saving_path)
output_dir.mkdir(parents=True, exist_ok=True)

# Get all subdirectories that contain a "gaze.csv"
dirs = sorted([
    d for d in Path(working_data_path).iterdir()
    if d.is_dir() and not d.name.startswith(".") and (d / "gaze.csv").exists()
])

if not dirs:
    raise ValueError("No recording directories found (no gaze.csv found).")

for rec_dir in dirs:
    process_recording(rec_dir, is_training_data, output_dir)



Cols with NaNs: []
Processed recording: 2026-04-15_10-47-38-63efffc7_Data -> 2026-04-15_10-47-38-63efffc7_Data.parquet


KeyboardInterrupt: 

In [9]:
# After all recordings, combine into a single Parquet.
combined = pl.scan_parquet(str(output_dir / "*.parquet"))
combined.sink_parquet(output_dir / "merged_output.parquet")
print("Combined Parquet saved as:", output_dir / "merged_output.parquet")

Combined Parquet saved as: ../data/Final Training Data/merged_output.parquet


I decided to create one CSV file, for each feature, that contains all the rows for each participant.

First I will combine most of the data into one CSV file and train a model with it. Then I will remove some data that does not feel relavent and see what works best. After I find the best comination of type of that I will try to extract more data out of the one i already have. For example insted of leaving the rows where the perticipant have blinked, I can remove it and calc the blinking rate for given frame. direc

I map the saccade Id onto the gaze data

Now I am combining all data into one CSV file. Firstly, I normalize the timestemps just to be sure they are in the same format as I will use them for sinhronization. I am using the gaze data as the base as it was capchured in 200 Hz and so it will be best if all other data is mapped to it. After normilizing the timestemps, I am mergeing the other files with the gaze data based on the timestamp or the Id of the event.

I am removing the rows where the "worn" column, in the gaze.csv, is equal to 0. Then I am dropping the "worn" column as it does not contain any useful information anymore

In [10]:
merged_df["label"].unique()

NameError: name 'merged_df' is not defined

In [ ]:
df = merged_df.group_by("recording id").agg(
    pl.col("label").unique().alias("labels")
)

for rec_id, labels in df.iter_rows():
    print(f"\nRecording: {rec_id}")
    print(labels)

In [ ]:
merged_df.filter(
    pl.col("label").is_null()
)

In [ ]:
merged_df.filter(
    pl.any_horizontal(pl.all().is_null())
)

In [ ]:
merged_df.select(
    pl.col("label").is_null().sum()
)

In [ ]:
labels_DF.filter(
    pl.col("start timestamp [ms]") == pl.col("end timestamp [ms]")
).collect()

In [ ]:
labels_DF.filter(
    pl.col("start timestamp [ms]") > pl.col("end timestamp [ms]")
).collect()